# mrm tutorial: from a controller template to a comparison, then the RL environment

This notebook walks through the controller workflow of the `mrm` package (07 P1-19) and the batched RL environment
(07 P1-22):

1. look at what a controller sees (the `ControllerSpec`);
2. build a controller from a template in `examples/controllers/`;
3. run it on the engine and read the metrics;
4. read the controller scorecard;
5. compare two controllers on the same worlds;
6. step the batched RL environment `mrm.sim.gym_env.MRMEnv`;
7. quantize a policy to fixed point and run it inside the compiled kernel.

**Every controller result below is conditional on the assumed thermal dynamics** (the g26 interim thermal model and
its assumed layout; docs/ARCHITECTURE.md section 7, rule 6). The numbers show how to use the tools, not how a chip
behaves.

Run time: about 10 s with a warm numba cache. The first run in a fresh environment compiles the plant kernel
(2-8 min, ~2.3 GB): run it once through `tools/memguard.py --heavy` (docs/ENGINE.md section 1). The notebook runs from
the repository root or from `examples/`.

In [1]:
import importlib.util, json, sys, time, warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np

from mrm import params
from mrm.control.api import ControllerSpec
from mrm.scenarios import library as lib
from mrm.sim import runner
from mrm.sim.metrics import RecordSpec

ROOT = Path.cwd() if (Path.cwd() / "mrm").is_dir() else Path.cwd().parent     # repository root
np.set_printoptions(precision=2, suppress=True, linewidth=110)

cfg = params.load("g26")                       # the default preset (a YAML path or overrides= work too)
spec = ControllerSpec.from_config(cfg, n_batch=1, nominal=cfg)
print(f"plant tick {spec.tick_s * 1e6:g} us; controller every {spec.period_ticks} ticks ({spec.rate_hz / 1e3:g} kHz); "
      f"{spec.samples_per_step} monitor ADC samples per step at most")
print("sensors  :", ", ".join(f.name for f in spec.layout.obs))
print("actuators:", ", ".join(f.name for f in spec.layout.act), "(tx.bias belongs to the scenario in g26)")
print("knowledge:", spec.knowledge, "(device numbers a controller may read from spec.nominal)")

plant tick 1 us; controller every 10 ticks (100 kHz); 3 monitor ADC samples per step at most
sensors  : tx.monitor.samples, tx.monitor.sample_ticks, tx.monitor.count, tx.monitor.latest, temp.die0, tx.ones_count, tx.bits, datapath.bitstat_above, rx.error_count, rx.error_count_valid, monitor_valid, adc_overrange, tx.heater.applied, tx.bias.applied
actuators: tx.heater, datapath.bitstat_threshold, monitor.pga_select (tx.bias belongs to the scenario in g26)
knowledge: ('line_grid', 'ring_to_line') (device numbers a controller may read from spec.nominal)


## 1. Build a controller from a template

`examples/controllers/template_python_controller.py` holds `MyIntegralLock`: an integral lock of the drop-port
monitor to a fraction of a *known* peak code, on the red (open-loop stable) side of the resonance, vectorized over
(B, N). We copy it and change one thing: instead of a peak we do not know, the set-point is the first monitor block
after the warm hand-off, so the controller holds the operating point it is handed (as the reference `FxpPi` does in
capture mode). On the red side the monitor rises when the resonance moves toward the line, so a monitor above the
set-point raises the heater (same sign as the template).

In [2]:
path = ROOT / "examples" / "controllers" / "template_python_controller.py"
if "template_python_controller" in sys.modules:         # re-running the cell: the template registers its name once
    tpl = sys.modules["template_python_controller"]
else:
    s = importlib.util.spec_from_file_location("template_python_controller", path)
    tpl = importlib.util.module_from_spec(s)
    sys.modules[s.name] = tpl
    s.loader.exec_module(tpl)
from mrm.control.baselines.pi_lock import block_inputs


class HoldLock(tpl.MyIntegralLock):
    # MyIntegralLock with the set-point captured from the first monitor block after the hand-off
    name = "hold_lock"

    @dataclass
    class Params:
        k_i_codes: float = 1000.0        # heater codes per step per unit of normalized error
        latency_ticks: int = 0

    def reset(self, spec):
        super().reset(spec)
        self.y_set = None

    def step(self, obs):
        if self.u is None:                                   # warm hand-off: start from the latched register
            self.u = np.asarray(obs["tx.heater.applied"], dtype=float).copy()
        self.y, ok = block_inputs(obs, self.y)               # block mean of the new samples; ok = new and valid
        if self.y_set is None:
            if not ok.all():                                 # no samples yet (the first edge): hold
                return {"tx.heater": np.rint(self.u).astype(np.int64)}
            self.y_set = self.y.copy()
        e = (self.y - self.y_set) / self.y_set
        self.u = np.where(ok, np.clip(self.u + self.params.k_i_codes * e, self.lo, self.hi), self.u)
        return {"tx.heater": np.rint(self.u).astype(np.int64)}


print(HoldLock.__mro__[1].__doc__)

u += k_i (y - f M_pk) / M_pk: on the red side dy/du < 0, so a monitor above the set-point raises the heater.


## 2. Run it

`runner.run` builds B worlds (here 4 sampled worlds: uncertainty and device variation drawn from the preset), starts
them warm (`heater_code="quasi_static"`: each ring at its oracle lock code; `settle=True`: the thermal state at its
steady state), and runs the controller at every controller edge on the Python path. The disturbance is the G26
slide-10 aggressor (A1) from its cool minimum: its power is calibrated so that ring 0 follows the printed A1
trajectory, ~4 K in 12 ms at up to ~600 degC/s here. The metrics are scored from 4 ms (`score_from_s`), about the
oracle's best-OMA operating point (`lock_reference="oracle"`). Rings whose quasi-static lock point the heater cannot
reach in a sampled world start at code 0 (a warning, silenced here); they are listed below, and their large errors in
the tables are that, not the controller.

In [3]:
scenario = lib.a1_slide10_aggressor(phase_s=lib.A1_PERIOD_S / 2)
common = dict(duration_s=12e-3, batch=4, seeds=3, uncertainty="sample", variation="sample", heater_code="quasi_static",
              settle=True, metrics={"lock_reference": "oracle"}, score_from_s=4e-3,
              record=[RecordSpec("detuning_m", 10, "mean")])
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    r_hold = runner.run(cfg, HoldLock(), scenario, **common)
print(f"{r_hold.manifest_inputs['n_ticks']} ticks x {r_hold.manifest_inputs['batch']} worlds in {r_hold.wall_s:.2f} s "
      f"({r_hold.manifest_inputs['controller_path']} path)")
no_code = np.asarray(r_hold.manifest_inputs["warm_start"]["no_code"])
print("rings without a lock code at the start (world, ring):", [tuple(int(v) for v in x) for x in np.argwhere(no_code)])

12000 ticks x 4 worlds in 1.10 s (python path)
rings without a lock code at the start (world, ring): [(0, 5), (0, 6), (2, 6), (3, 7)]


## 3. Read the metrics

`RunResult.metrics` holds the streaming metrics of the kernel (07 P0-8; per ring (B, N) arrays unless noted),
`trace` the recorded Truth fields (here the detuning, 10 us means), `truth_final` the full Truth at the end,
`manifest_inputs` what a run manifest needs. Truth never reaches a controller.

In [4]:
m = r_hold.metrics
print("some metric keys:", ", ".join(k for k in list(m)[:14]))
for k in ("rms_lock_error_pm", "max_abs_lock_error_pm", "lock_reference_pm", "oma_penalty_db"):
    print(f"{k:24s} world 0: {np.asarray(m[k])[0]}")
tr = r_hold.trace["detuning_m@10:mean"]
print(f"trace detuning: {tr['value'].shape} (samples, worlds, rings), windows ending at {tr['t_s'][0] * 1e6:.0f} us "
      f"... {tr['t_s'][-1] * 1e3:.0f} ms")
print("diagnostics:", {k: int(np.sum(v)) for k, v in r_hold.diagnostics.items() if k in ("selfheat_not_converged",
      "commands_clipped", "pipeline_overflow")})

some metric keys: duration_s, max_abs_detuning_pm, mean_detuning_pm, rms_detuning_pm, max_abs_lock_error_pm, rms_lock_error_pm, lock_reference_pm, time_above_s, locked_final, ever_locked, excursion_count, excursion_max_s, time_unlocked_s, excursion_run_hist_log2
rms_lock_error_pm        world 0: [  56.47   56.85   56.1    57.17   48.2   572.95 1236.62   47.2 ]
max_abs_lock_error_pm    world 0: [ 141.55  143.81  143.94  145.85  129.65  651.62 1331.26  127.97]
lock_reference_pm        world 0: [113.77 116.52 118.56 119.2  113.58 109.31 108.95 113.21]
oma_penalty_db           world 0: [ 0.18  0.16  0.13  0.15  0.06 12.12 19.96  0.05]
trace detuning: (1200, 4, 8) (samples, worlds, rings), windows ending at 10 us ... 12 ms
diagnostics: {'selfheat_not_converged': 0, 'pipeline_overflow': 0, 'commands_clipped': 0}


## 4. The controller scorecard

`tests/validation/test_controllers_on_engine.py` runs every registered controller on the engine (g26 nominal world,
warm or cold start, steady 40 ms and the full A1 period) and writes `docs/handoff/controller_scorecard.md` / `.json`
(the conditions are at the top of the Markdown file). A few rows:

In [5]:
card = json.loads((ROOT / "docs" / "handoff" / "controller_scorecard.json").read_text())
print(f"{'controller':16s} {'path':9s} {'locked':>6s} {'jitter rms (pm)':>15s} {'A1 max|e| ring 0':>17s} {'rings 1-7':>10s}")
for name in ("pi_lock", "pi_lock_c", "dither_lockin", "feedforward_pi", "kalman_lqg", "mpc_c"):
    c = card[name]
    st, a1 = c["steady"], c.get("a1") or {}
    mx = a1.get("max_pm") or [np.nan] * 8
    print(f"{name:16s} {c['path']:9s} {sum(st['locked']):>4d}/8 {np.median(st['rms_pm']):15.2f} {mx[0]:17.1f} "
          f"{max(mx[1:]):10.1f}")

controller       path      locked jitter rms (pm)  A1 max|e| ring 0  rings 1-7
pi_lock          python       8/8            0.01             348.5        6.0
pi_lock_c        compiled     8/8            0.01             348.5        6.0
dither_lockin    python       4/8            8.65            1337.0      560.6
feedforward_pi   python       8/8            0.09            5102.6     5004.5
kalman_lqg       python       8/8            0.00             363.6        6.5
mpc_c            compiled     8/8            0.00             363.6        6.5


## 5. Compare two controllers

The registered `pi_lock` (map the peak, then a PI at 0.7 of it) against the template-derived `HoldLock`, on the same
four worlds, seeds, start and scenario. Because the two controllers lock at different points (`pi_lock` at its 0.7
fraction, `HoldLock` at the hand-off point), the first column is the tracking error about each ring's own operating
point at the scoring start: the largest |d(t) - d(4 ms)| over the scored window. The OMA penalty compares each
ring's relOMA with the oracle's at the same instant.

In [6]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    r_pi = runner.run(cfg, "pi_lock", scenario, **common)
    r_open = runner.run(cfg, None, scenario, **common)


def tracking_pm(r, t0_s=4e-3):
    tr = r.trace["detuning_m@10:mean"]
    i0 = int(np.searchsorted(tr["t_s"], t0_s))
    d = tr["value"] * 1e12
    return np.abs(d[i0:] - d[i0]).max(axis=0)                # (B, N)


print(f"{'controller':10s} {'tracking, median ring (pm)':>27s} {'worst ring (pm)':>16s} {'OMA penalty, median (dB)':>25s} "
      f"{'wall (s)':>9s}")
for name, r in (("open loop", r_open), ("HoldLock", r_hold), ("pi_lock", r_pi)):
    t = tracking_pm(r)
    pen = np.asarray(r.metrics["oma_penalty_db"])
    print(f"{name:10s} {np.median(t):27.2f} {t.max():16.1f} {np.nanmedian(pen):25.3f} {r.wall_s:9.2f}")
print("ring 0 (the A1 victim), per world:")
for name, r in (("open loop", r_open), ("HoldLock", r_hold), ("pi_lock", r_pi)):
    print(f"  {name:10s}", tracking_pm(r)[:, 0])

controller  tracking, median ring (pm)  worst ring (pm)  OMA penalty, median (dB)  wall (s)
open loop                        15.40            175.1                     0.144      0.34
HoldLock                          0.27             48.9                     0.056      1.10
pi_lock                           0.66            112.2                     0.183      1.05
ring 0 (the A1 victim), per world:
  open loop  [152.43 175.07  34.97  73.68]
  HoldLock   [ 0.93  0.51 34.91 24.1 ]
  pi_lock    [ 0.89  0.73  0.34 67.76]


Four worlds, one scenario and an 8 ms window are a demonstration, not a ranking. To compare controllers properly,
run each on many sampled worlds (`batch=64`, a compiled controller for speed), on
the named corners of the uncertainty set, and report the nominal result, the worst named corner and quantiles over
the uncertainty set (docs/ARCHITECTURE.md section 7, rule 6). `mrm.sim.experiment` has the sweep / Monte Carlo /
compare tools.

## 6. The batched RL environment

`MRMEnv` is a `gymnasium.vector.VectorEnv` over the plant batch: B worlds of one bank, one kernel call per step, no
subprocess workers. One step is one controller period. The action is a bounded delta of each heater code (and of the
bias code when the controller owns the bias); the observation is the last K monitor sample blocks, the applied codes
and the temperature-sensor codes, each normalized by a power of two; the reward comes from the kernel's metric
accumulators (from Truth), and Truth itself only appears in `info["truth"]`. Every reset draws new worlds.

In [7]:
from mrm.sim.gym_env import MRMEnv

env = MRMEnv("g26", num_envs=8, max_episode_steps=200)
print(env)
print("observation:", env.single_observation_space.shape, [(c.name, c.shape) for c in env.obs_layout.channels])
print("action     :", env.single_action_space, "->", env.action_codec.names(), "+-", env.action_codec.channels[0][3],
      "codes per step")
rng = np.random.default_rng(0)
for label, policy in (("hold (zero action)", lambda o: np.zeros(env.action_space.shape, np.float32)),
                      ("random +-10 %", lambda o: rng.uniform(-0.1, 0.1, env.action_space.shape).astype(np.float32))):
    obs, info = env.reset(seed=1)
    t0, rew = time.perf_counter(), []
    for k in range(100):
        obs, r, term, trunc, info = env.step(policy(obs))
        rew.append(r)
    dt = time.perf_counter() - t0
    print(f"{label:20s} mean reward {np.mean(rew):7.3f}; step rms lock error {np.median(info['truth']['step_rms_lock_error_m']) * 1e12:6.2f} pm "
          f"(median ring); {100 * env.num_envs / dt:,.0f} env steps/s at B = {env.num_envs}")
print("info['truth'] keys:", sorted(info["truth"]))
print("worlds of the last reset: device seeds", env.seeds["device"], "...", env.seeds["device"] + env.num_envs - 1)

MRMEnv(B=8, N=8, obs=113, act=8, frame_skip=10 ticks, start='warm')
observation: (113,) [('tx.monitor.samples', (8, 4, 3)), ('tx.heater.applied', (8,)), ('tx.bias.applied', (8,)), ('temp.die0', (1,))]
action     : Box(-1.0, 1.0, (8,), float32) -> ('tx.heater',) +- 256 codes per step
hold (zero action)   mean reward  -4.100; step rms lock error  17.38 pm (median ring); 7,289 env steps/s at B = 8
random +-10 %        mean reward  -4.209; step rms lock error  21.83 pm (median ring); 7,028 env steps/s at B = 8
info['truth'] keys: ['detuning_m', 'heater_code', 'line_on', 'lock_error_m', 'lock_reference_m', 'oracle_reloma', 'reloma', 'reward_per_ring', 'step_rms_lock_error_m', 't_ring_k']
worlds of the last reset: device seeds 508082488 ... 508082495


## 7. A fixed-point policy on the engine

A policy that cannot be quantized is not an FPGA controller (07 P1-22). `mrm.control.fxp_mlp.quantize_mlp` chooses
every fixed-point format of an MLP (the accumulator width is proven never to overflow), `error_bound` bounds the
float-vs-fixed difference, and `MRMEnv.compiled_policy` runs the quantized network inside the compiled kernel on the
same observation, built with integer shifts. Here a small untrained network (so its actions are arbitrary) is run
both ways: through the env and as a compiled controller; the commands, and so the trajectories, are identical.

In [8]:
from mrm.control import fxp_mlp
from mrm.sim.gym_env import fxp_policy_action

D, A = env.obs_layout.size, env.action_codec.size
rng = np.random.default_rng(7)
ws = [rng.normal(0, 0.4 / np.sqrt(D), (16, D)), rng.normal(0, 0.5, (A, 16))]
bs = [rng.normal(0, 0.1, 16), rng.normal(0, 0.1, A)]
acts = ["relu", "hardtanh"]
mlp = fxp_mlp.quantize_mlp(ws, bs, acts)                       # Q0.15 inputs, 16-bit weights and activations
print({k: v for k, v in mlp.describe().items() if k != "layers"})
for i, l in enumerate(mlp.describe()["layers"]):
    print(f"  layer {i}: {l['n_in']:3d} -> {l['n_out']:2d}, weights {l['w_fmt']}, accumulator {l['acc_bits']} bits, "
          f"out {l['out_fmt']} ({l['activation']})")
eb = fxp_mlp.error_bound(mlp, ws, bs, acts, input_exact=True)
print(f"float-vs-fixed output error bound over the whole input range: {eb.bound.max():.2e} (saturation free: "
      f"{eb.saturation_free}); {mlp.resources()['macs']} MACs per step")

small = MRMEnv("g26", num_envs=2, max_episode_steps=None, max_delta_heater_codes=16)
seeds = {"device": 11, "uncertainty": 11, "noise": 12, "scenario": 13}
obs, _ = small.reset(options={"seeds": seeds})
for k in range(200):
    obs, *_ = small.step(fxp_policy_action(mlp, obs))
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    r_c = runner.run(cfg, small.compiled_policy(mlp), None, duration_s=200 * small.step_s, batch=2, seeds=seeds,
                     heater_code="quasi_static", settle=True, metrics={"lock_reference": "oracle"}, histogram=False)
same = np.array_equal(small.plant.truth()["detuning_m"], r_c.truth_final["detuning_m"])
print(f"env vs compiled policy on the engine ({r_c.manifest_inputs['controller_path']} path): final detuning "
      f"identical: {same}; heater codes identical: "
      f"{np.array_equal(small.plant.truth()['heater_code'], r_c.truth_final['heater_code'])}")

{'in_fmt': 'Q0.15', 'rounding': 'half_even', 'overflow': 'saturate', 'proven_no_acc_overflow': True}
  layer 0: 113 -> 16, weights Q0.15, accumulator 34 bits, out Q3.12 (relu)
  layer 1:  16 ->  8, weights Q1.14, accumulator 33 bits, out Q1.14 (hardtanh)
float-vs-fixed output error bound over the whole input range: 8.01e-03 (saturation free: True); 1936 MACs per step
env vs compiled policy on the engine (compiled path): final detuning identical: True; heater codes identical: True


## Where to go next

- Train a policy with any RL library on `MRMEnv` (or `gymnasium.make_vec("mrm/MRM-v0", num_envs=B,
  vectorization_mode="vector_entry_point")`); give the final layer a hardtanh (the env clips actions to [-1, 1], and so does
  `compiled_policy`).
- Quantize it with `quantize_mlp`, check `error_bound`, deploy it with `compiled_policy` (Monte Carlo at kernel speed,
  the binary bridge and golden vectors of 07 P1-21), and export its weights with `mlp.memh()` for `$readmemh`.
- docs/ENGINE.md explains the engine; docs/handoff/rl.md the environment (design, measured speed, open items).